# Hotel Reservation Cancellation Prediction

This notebook implements a complete machine learning pipeline to predict whether a hotel reservation will be cancelled, based on booking details and guest behaviour. The 21 steps below cover the full journey from raw data to a deployed, evaluated model.

## Step 1: Import Libraries

All required libraries are imported here. `scikit-learn` provides the modelling, pipeline, and evaluation tools. `pandas` and `numpy` handle data manipulation, and `matplotlib` and `seaborn` support exploratory visualisation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries imported successfully.')

## Step 2: Load Dataset

The dataset is loaded from CSV. If running in Google Colab, upload the file using the file browser or `files.upload()` before executing this cell.

In [ ]:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('Hotel_Reservations.csv')
print(f'Loaded: {df.shape}')
print(df.columns.tolist())

## Step 3: Inspect Dataset

A structural review confirms column types, the presence of the target variable, and an initial view of the data distribution.

In [ ]:
print(df.dtypes)
display(df.head())
print('\nTarget distribution:')
print(df['booking_status'].value_counts())

## Step 4: Define Binary Target

The `booking_status` column is mapped to a binary integer target:
- `Canceled` → 1 (the positive class)
- `Not_Canceled` → 0

This is a standard binary classification framing where identifying cancellations is the primary objective.

In [ ]:
df['target'] = (df['booking_status'] == 'Canceled').astype(int)
print('Target distribution:')
print(df['target'].value_counts())
print(f'Cancellation rate: {df["target"].mean():.2%}')

## Step 5: Clean Data

The `Booking_ID` column is an identifier with no predictive value and is removed. The original `booking_status` column is also dropped, as the target has been derived from it.

In [ ]:
df = df.drop(columns=['Booking_ID', 'booking_status'])
print(f'Shape after cleaning: {df.shape}')

## Step 6: Handle Missing Values

The dataset is inspected for missing values. No imputation is required here, as the dataset is complete, but the preprocessing pipeline includes imputers as a precaution for robustness during deployment.

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

## Step 7: Feature Engineering

Three derived features are constructed to capture aggregate patterns not directly available in the raw columns:
- `total_nights`: total stay duration (weekend plus weekday nights)
- `total_guests`: combined guest count (adults plus children)
- `cancellation_rate`: proportion of prior bookings that were cancelled

In [ ]:
df['total_nights'] = df['no_of_weekend_nights'] + df['no_of_week_nights']
df['total_guests']  = df['no_of_adults'] + df['no_of_children']

total_prior = df['no_of_previous_cancellations'] + df['no_of_previous_bookings_not_canceled']
df['cancellation_rate'] = df['no_of_previous_cancellations'] / total_prior.replace(0, np.nan)
df['cancellation_rate'] = df['cancellation_rate'].fillna(0)

print('Engineered features:')
display(df[['total_nights','total_guests','cancellation_rate']].describe())

## Step 8: Exploratory Analysis

Key relationships between features and the cancellation outcome are examined visually and numerically.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Key Feature Relationships with Cancellation', fontsize=13, fontweight='bold')

# Lead time
df.boxplot(column='lead_time', by='target', ax=axes[0],
           boxprops=dict(color='#374151'), medianprops=dict(color='#2563eb'))
axes[0].set_title('Lead Time by Outcome')
axes[0].set_xticklabels(['Not Cancelled','Cancelled'])
plt.sca(axes[0]); plt.title('Lead Time by Outcome')

# Average price
df.boxplot(column='avg_price_per_room', by='target', ax=axes[1],
           boxprops=dict(color='#374151'), medianprops=dict(color='#2563eb'))
axes[1].set_title('Avg Price by Outcome')
axes[1].set_xticklabels(['Not Cancelled','Cancelled'])
plt.sca(axes[1]); plt.title('Avg Price by Outcome')

# Special requests
df.groupby('target')['no_of_special_requests'].mean().plot(
    kind='bar', ax=axes[2], color=['#16a34a','#dc2626'])
axes[2].set_title('Avg Special Requests')
axes[2].set_xticklabels(['Not Cancelled','Cancelled'], rotation=0)

plt.tight_layout()
plt.show()

print('\nCancellation rate by market segment:')
print(df.groupby('market_segment_type')['target'].mean().round(3).sort_values(ascending=False))

## Step 9: Select Features

Features are selected based on domain relevance and their predictive potential. The `Booking_ID` and original target column have already been removed.

In [ ]:
FEATURE_COLS = [
    'no_of_adults','no_of_children','no_of_weekend_nights','no_of_week_nights',
    'type_of_meal_plan','required_car_parking_space','room_type_reserved',
    'lead_time','arrival_year','arrival_month','arrival_date',
    'market_segment_type','repeated_guest',
    'no_of_previous_cancellations','no_of_previous_bookings_not_canceled',
    'avg_price_per_room','no_of_special_requests',
    'total_nights','total_guests','cancellation_rate',
]
X = df[FEATURE_COLS].copy()
y = df['target'].copy()
print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')

## Step 10: Define Column Types

Features are categorised as categorical or numerical to apply appropriate preprocessing transformations within the `ColumnTransformer`.

In [ ]:
CATEGORICAL_COLS = ['type_of_meal_plan','room_type_reserved','market_segment_type']
NUMERICAL_COLS = [
    'no_of_adults','no_of_children','no_of_weekend_nights','no_of_week_nights',
    'required_car_parking_space','lead_time','arrival_year','arrival_month',
    'arrival_date','repeated_guest','no_of_previous_cancellations',
    'no_of_previous_bookings_not_canceled','avg_price_per_room',
    'no_of_special_requests','total_nights','total_guests','cancellation_rate',
]
print(f'Categorical: {CATEGORICAL_COLS}')
print(f'Numerical:   {NUMERICAL_COLS}')

## Step 11: Train-Test Split

A stratified 80/20 split is applied to maintain class proportions in both partitions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')
print(f'Train balance:\n{y_train.value_counts(normalize=True).round(3)}')

## Step 12: Preprocessing Pipeline

A `ColumnTransformer` applies median imputation and standard scaling to numerical features, and constant imputation with one-hot encoding to categorical features. `handle_unknown='ignore'` prevents errors on unseen categories at inference time.

In [ ]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', num_transformer, NUMERICAL_COLS),
    ('cat', cat_transformer, CATEGORICAL_COLS),
])
print('Preprocessing pipeline constructed.')

## Step 13: Train Logistic Regression

Logistic Regression is used as the interpretable baseline. It assumes a linear relationship between features and the log-odds of cancellation, and is fast to train and easy to inspect.

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_lr, target_names=['Not Cancelled','Cancelled']))

## Step 14: Train Decision Tree

The Decision Tree offers full interpretability through its rule structure. A maximum depth of 10 is imposed to limit overfitting, though the model remains more prone to variance than the ensemble approach.

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced')),
])
dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_dt, target_names=['Not Cancelled','Cancelled']))

## Step 15: Train Random Forest

Random Forest aggregates predictions across many decision trees trained on bootstrapped subsets. This reduces overfitting and produces more stable, generalisable predictions than a single tree.

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1
    )),
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_rf, target_names=['Not Cancelled','Cancelled']))

## Step 16: Evaluate Models

All three models are evaluated using accuracy, precision, recall, F1-score, confusion matrices, and three-fold stratified cross-validation.

In [ ]:
def eval_model(name, yt, yp):
    return {'Model':name,
            'Accuracy':round(accuracy_score(yt,yp),4),
            'Precision':round(precision_score(yt,yp,zero_division=0),4),
            'Recall':round(recall_score(yt,yp,zero_division=0),4),
            'F1-Score':round(f1_score(yt,yp,zero_division=0),4)}

results = pd.DataFrame([
    eval_model('Logistic Regression', y_test, y_pred_lr),
    eval_model('Decision Tree',       y_test, y_pred_dt),
    eval_model('Random Forest',       y_test, y_pred_rf),
])
display(results)

# Confusion matrices
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax,(name,yp) in zip(axes,[('Logistic Regression',y_pred_lr),('Decision Tree',y_pred_dt),('Random Forest',y_pred_rf)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test,yp),display_labels=['Not Cancelled','Cancelled']).plot(ax=ax,colorbar=False,cmap='Blues')
    ax.set_title(name)
plt.tight_layout(); plt.show()

# Cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for name, pipe in [('LR',lr_pipeline),('DT',dt_pipeline),('RF',rf_pipeline)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    print(f'{name}: {scores.round(4)} | Mean: {scores.mean():.4f}')

## Step 17: Hyperparameter Tuning

`GridSearchCV` is applied to the Random Forest using a compact parameter grid. The grid focuses on the most impactful regularisation parameters to balance performance with computational efficiency.

In [ ]:
param_grid = {
    'classifier__n_estimators':      [100,150],
    'classifier__max_depth':         [10,12],
    'classifier__min_samples_split': [10],
    'classifier__min_samples_leaf':  [5],
}
rf_tuned = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(random_state=42, class_weight='balanced')),
])
grid_search = GridSearchCV(rf_tuned, param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print(f'Best params: {grid_search.best_params_}')
print(f'Best F1:     {grid_search.best_score_:.4f}')
best_model  = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=['Not Cancelled','Cancelled']))

## Step 18: Compare Models

A final summary table compares all model configurations on the held-out test set.

In [ ]:
final = pd.DataFrame([
    eval_model('Logistic Regression',   y_test, y_pred_lr),
    eval_model('Decision Tree',         y_test, y_pred_dt),
    eval_model('Random Forest',         y_test, y_pred_rf),
    eval_model('Random Forest (Tuned)', y_test, y_pred_best),
])
display(final)

# Bar chart
metrics = ['Accuracy','Precision','Recall','F1-Score']
x = np.arange(len(metrics)); w = 0.2
fig, ax = plt.subplots(figsize=(12,5))
colours = ['#6b7280','#f59e0b','#3b82f6','#16a34a']
for i,(_,row) in enumerate(final.iterrows()):
    ax.bar(x+i*w, row[metrics].values, w, label=row['Model'], color=colours[i])
ax.set_xticks(x+w*1.5); ax.set_xticklabels(metrics)
ax.set_ylabel('Score'); ax.set_ylim(0,1)
ax.set_title('Model Performance Comparison')
ax.legend(); plt.tight_layout(); plt.show()

## Step 19: Select Final Model

The tuned Random Forest is selected as the final model based on its consistent superiority across all evaluation metrics, particularly recall on the cancelled class.

In [ ]:
print('Final model: Tuned Random Forest')
print(f'Accuracy: {accuracy_score(y_test,y_pred_best):.4f}')
print(f'F1-Score: {f1_score(y_test,y_pred_best):.4f}')

## Step 20: Feature Importance

Feature importances from the fitted Random Forest indicate which inputs have the greatest influence on the model's predictions.

In [ ]:
rf_clf = best_model.named_steps['classifier']
ohe_names = (best_model.named_steps['preprocessor']
             .named_transformers_['cat'].named_steps['encoder']
             .get_feature_names_out(CATEGORICAL_COLS).tolist())
all_names   = NUMERICAL_COLS + ohe_names
importances = pd.Series(rf_clf.feature_importances_, index=all_names)
top15       = importances.sort_values(ascending=False).head(15)
print(top15.round(4).to_string())

fig, ax = plt.subplots(figsize=(10,6))
top15.sort_values().plot(kind='barh', ax=ax, color='#3b82f6')
ax.set_title('Top 15 Feature Importances — Tuned Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.show()

## Step 21: Final Conclusions

The modelling process is summarised with key findings and deployment notes.

In [ ]:
print("""
FINAL CONCLUSIONS
=================
Three models were evaluated on the hotel reservation cancellation task.

Logistic Regression delivered a solid baseline.
Decision Tree was interpretable but showed signs of overfitting.
Random Forest, after tuning, achieved the strongest overall performance.

The most important features were lead time, average room price,
number of special requests, and the derived cancellation rate.

The tuned Random Forest is deployed in the Streamlit application.

Limitations:
- Data covers two calendar years from one hotel group.
- Model drift should be expected if booking behaviour shifts.
- Predictions are probabilistic and should supplement, not replace,
  operational judgement.
""")